# Week 05: Images

Images are self-contained datasets....

## PIL

The Python Image Library has some great tools and functionality for working with images.

[Reference and Documentation](https://pillow.readthedocs.io/en/stable/)

We just have to run the following cell to import it, and some of its sub-modules.

In [ ]:
from PIL import Image as PImage, ImageFilter as PImageFilter

## Review: Opening and displaying an image

In [ ]:
img = PImage.open("./data/image/hog.jpg")
display(img)

## Review: Getting pixels

In [ ]:
# getting image pixel list

img = PImage.open("./data/image/hog.jpg")
pxs = list(img.getdata())

display(len(pxs))
display(pxs[0])
display(pxs[:5])

## Review: RGB color space

In [ ]:
himg = PImage.open("./data/image/hog.jpg")
himg_pxs = list(himg.getdata())

# build array of new pixel values
redpxs = []
for r,g,b in himg_pxs:
  redpxs.append((r, 0, 0))

himg.putdata(redpxs)
display(himg)

## Experiment with this structure

Can you modify the code to remove the green channel ?

What happens if you swap color channels ?

In [ ]:
# TODO: experiment here

## Filter: Keep only green pixels

This is different from keeping the green channel.

The green channel is also used by colors that aren't _green_, like white and yellow.

Here we want to detect when a pixel is actually green.

In [ ]:
# TODO: keep green pixels, otherwise make it black

himg = PImage.open("./data/image/hog.jpg")
himg_pxs = list(himg.getdata())

# build array of new pixel values
greenpxs = []
for r,g,b in himg_pxs:
  pass

himg.putdata(greenpxs)
display(himg)

## Remove color

Something else we might want to do is remove color information and only keep brightness/luminosity information.

So, turn the image into grayscale.

The simplified way for doing this is to just average the $3$ channels and put that single luminosity value back as the pixel's `R`, `G` and `B` values.

$\displaystyle \text{luminosity} = \frac{R + G + B}{3}$

This is a good way to estimate the brightness of each pixel: brighter pixels will be white and darker pixels will be black.

In [ ]:
img = PImage.open("./data/image/hog.jpg")
pxs = list(img.getdata())

npxs = []
for r,g,b in pxs:
  l = int((r + g + b) // 3)
  npxs.append((l, l, l))

img.putdata(npxs)
img

## Filter effect

Let's say we want to replicate this effect from *Schindler's List* to highlight a specific color in an image.

<img src="./imgs/red-coat-filter.jpg" width="720px">

The logic could be something like: if pixel is red, keep it, otherwise turn pixel into greyscale.

Let's replicate something like this for the hedgehog image, but keep green pixels.

We'll loop through all pixels. If a pixel's `green` channel value is greater than both its `red` and `blue` values, we'll keep the original values. If it's not a green pixel, we'll make it gray.

In [ ]:
# TODO: detect green pixels
# TODO: change green pixels to grayscale

### What about yellow ?

What if we want to keep the yellows ?

Yellow is not a channel, so we have to stitch together some logic to detect yellow pixels.

A pixel is yellow if its `reg` and `green` channels are similar, `red` is slightly greater than `green` and both are greater than `blue`, which is low.

In [ ]:
# TODO: keep yellow pixels, make rest grayscale
#       (240, 200, 0) are good values to use for yellow 

## Distances

It gets really hard to describe colors and filters this way for colors that aren't so easily described in terms of `red`, `green` and `blue` proportions.

Another way to think about this is to pick `RGB` values for a yellow color and then see how similar our pixels are to that value.

We can use the idea of distance to calculate this. Either measure and add individual distance for each channel in $1D$, or use the formula for $3D$ distance.

### 1-dimension (easy):

$\text{abs}(x_0 - x_1)$

### 2-dimensions (euclidean distance):
$\sqrt{\left(x_0 - x_1\right)^{2} + \left(y_0 - y_1\right)^{2}}$

### color distance is a 3-dimensional euclidean distance:
$\sqrt{\left(R_0 - R_1\right)^{2} + \left(G_0 - G_1\right)^{2} + \left(B_0 - B_1\right)^{2}}$

### We can implement functions to help us

The `color_distance()` function measures the distance between two colors in $3D$ space.

The `color_similar()` function measure if $2$ colors are similar using single-channel comparisons and pre-determined thresholds.

In [ ]:
def color_distance(c0, c1):
  r0,g0,b0 = c0
  r1,g1,b1 = c1
  return ((r0 - r1)**2 + (g0 - g1)**2 + (b0 - b1)**2)**0.5

def color_similar(c0, c1, ts):
  r0,g0,b0 = c0
  r1,g1,b1 = c1
  rt,gt,bt = ts
  return (abs(r0 - r1) < rt) and (abs(g0 - g1) < gt) and (abs(b0 - b1) < bt)

In [ ]:
# TODO: repeat yellow filter using functions

## Analyzing images

One last operation that is useful when manipulating/filtering images like this, is to count how many pixels fit a certain description.

Knowing how much of our image is green or yellow can help us determine what is in the image.

### Counting pixels

We can do this by first filtering our image and then counting how many pixels are not gray.

Or, we can use the `color_distance()` function to detect a color and increment a counter when we see pixels of that color.

As a final step, it's useful to divide this counter by the size of the image so we get a relative value for our color counter. This way images of different sizes will give similar results if their content is similar.

In [ ]:
# TODO: count yellow pixels
# TODO: count white pixels

## Exercise

### Download Data

In [ ]:
!wget -qO- https://github.com/PSAM-5005-2026S-A/5005-utils/releases/latest/download/forest-sat.tar.gz | tar xz

### Detect deforestation based on satellite imagery

1\. Look into `./data/image/forest-sat`<br>
&emsp;a. First $5$ digits are id for a location<br>
&emsp;b. Last $4$ digits are the year the photo was taken<br>
2\. Pick a location id that starts with $0$<br>
3\. Open the first image for the chosen location (from $1986$)<br>
&emsp;a. Determine a value for its green pixels<br>
&emsp;b. Count the amount of green pixels<br>
4\. Open the last image for the chosen location (from $2020$)<br>
&emsp;a. Determine a value for its green pixels<br>
&emsp;b. Count the amount of green pixels<br>
5\. Adjust color values and thresholds<br>
6\. Could you detect deforestation before $2020$ ?

#### Some tips
1\. Consider creating $2$ functions, one for filtering an image for a given color (image $\to$ image), and another for counting the number of non-black pixels in an image (image $\to$ number).<br>
2\. Consider shrinking the images: this speeds up processing and a size change won't affect the relative amount of green on an image.<br>
3\. Consider [blurring](https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html) the images: this makes color tones more similar to each other and simplifies filtering.<br>
4\. Consider automating the process of determining the green values for each image.

In [ ]:
# TODO: detect deforestation